In [8]:
import nfl_data_py as nfl
import os

# Set path relative to project root, not notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_path = os.path.join(project_root, "data", "raw")
processed_path = os.path.join(project_root, "data", "processed")

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")

Raw data path: /workspaces/nfl-4th-down-analysis/data/raw
Processed data path: /workspaces/nfl-4th-down-analysis/data/processed


In [9]:
seasons = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

for season in seasons:
    print(f"Pulling {season}...")
    df = nfl.import_pbp_data([season])
    df.to_parquet(os.path.join(raw_path, f"pbp_{season}.parquet"), index=False)
    print(f"{season} saved.")
    del df

print("All seasons saved.")

Pulling 2016...
2016 done.
Downcasting floats.
2016 saved.
Pulling 2017...
2017 done.
Downcasting floats.
2017 saved.
Pulling 2018...
2018 done.
Downcasting floats.
2018 saved.
Pulling 2019...
2019 done.
Downcasting floats.
2019 saved.
Pulling 2020...
2020 done.
Downcasting floats.
2020 saved.
Pulling 2021...
2021 done.
Downcasting floats.
2021 saved.
Pulling 2022...
2022 done.
Downcasting floats.
2022 saved.
Pulling 2023...
2023 done.
Downcasting floats.
2023 saved.
Pulling 2024...
2024 done.
Downcasting floats.
2024 saved.
Pulling 2025...
2025 done.
Downcasting floats.
2025 saved.
All seasons saved.


In [10]:
import pandas as pd

files = os.listdir(raw_path)
print(f"Files in data/raw: {sorted(files)}")

# Load one season to confirm it reads correctly
test = pd.read_parquet(os.path.join(raw_path, "pbp_2023.parquet"))
print(f"\n2023 shape: {test.shape}")
print(f"Columns: {test.columns.tolist()[:10]}...")
del test

Files in data/raw: ['pbp_2016.parquet', 'pbp_2017.parquet', 'pbp_2018.parquet', 'pbp_2019.parquet', 'pbp_2020.parquet', 'pbp_2021.parquet', 'pbp_2022.parquet', 'pbp_2023.parquet', 'pbp_2024.parquet', 'pbp_2025.parquet']

2023 shape: (49665, 396)
Columns: ['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam']...


In [11]:
cols = [
    'play_id', 'game_id', 'season', 'week',
    'posteam', 'defteam', 'home_team', 'away_team',
    'home_coach', 'away_coach', 'posteam_type',
    'down', 'ydstogo', 'yardline_100',
    'score_differential', 'game_seconds_remaining',
    'qtr', 'goal_to_go',
    'play_type', 'fourth_down_converted', 'fourth_down_failed',
    'epa', 'wp', 'wpa',
    'field_goal_result', 'punt_attempt', 'field_goal_attempt'
]

dfs = []
for season in range(2016, 2026):
    df = pd.read_parquet(os.path.join(raw_path, f"pbp_{season}.parquet"), columns=cols)
    df = df[df['down'] == 4]
    df = df[df['play_type'].isin(['pass', 'run', 'punt', 'field_goal'])]
    dfs.append(df)
    print(f"{season}: {len(df)} 4th down plays")

fourth_downs = pd.concat(dfs, ignore_index=True)

fourth_downs['coach'] = fourth_downs.apply(
    lambda row: row['home_coach'] if row['posteam_type'] == 'home' else row['away_coach'],
    axis=1
)
fourth_downs['decision'] = fourth_downs['play_type'].map({
    'pass': 'go',
    'run': 'go',
    'punt': 'punt',
    'field_goal': 'field_goal'
})

fourth_downs.to_parquet(os.path.join(processed_path, "fourth_downs.parquet"), index=False)
print(f"\nTotal shape: {fourth_downs.shape}")
print(f"\nDecision breakdown:\n{fourth_downs['decision'].value_counts()}")

2016: 3891 4th down plays
2017: 4027 4th down plays
2018: 3782 4th down plays
2019: 3801 4th down plays
2020: 3596 4th down plays
2021: 3982 4th down plays
2022: 4071 4th down plays
2023: 4222 4th down plays
2024: 3998 4th down plays
2025: 3997 4th down plays

Total shape: (39367, 29)

Decision breakdown:
decision
punt          22519
field_goal     9790
go             7058
Name: count, dtype: int64


In [12]:
# Add a column for the coach who made the decision
fourth_downs['coach'] = fourth_downs.apply(
    lambda row: row['home_coach'] if row['posteam_type'] == 'home' else row['away_coach'],
    axis=1
)

# Add a column for the decision made
fourth_downs['decision'] = fourth_downs['play_type'].map({
    'pass': 'go',
    'run': 'go',
    'punt': 'punt',
    'field_goal': 'field_goal'
})

# Save to processed
output_file = os.path.join(processed_path, "fourth_downs.parquet")
fourth_downs.to_parquet(output_file, index=False)
print(f"Saved to {output_file}")
print(f"Shape: {fourth_downs.shape}")
print(f"\nDecision breakdown:\n{fourth_downs['decision'].value_counts()}")

Saved to /workspaces/nfl-4th-down-analysis/data/processed/fourth_downs.parquet
Shape: (39367, 29)

Decision breakdown:
decision
punt          22519
field_goal     9790
go             7058
Name: count, dtype: int64
